# WiDS 2025

## Step 2. AutoML - EDA

In [1]:
import h2o
from h2o.automl import H2OAutoML

import shap

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321.

/Applications/anaconda3/envs/wids-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 connected.


H2O_cluster_uptime:,5 hours 39 mins
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,2 months and 17 days
H2O_cluster_name:,H2O_from_python_fujinhuizi_so03bv
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [2]:
import glob
import os
import sys
import pandas as pd

In [3]:
current_working_directory = os.getcwd()
home_dir = os.path.abspath(os.path.join(current_working_directory, os.pardir))
data_path = os.path.join(home_dir, 'data/')

In [ ]:
path_small_input = os.path.join(data_path, 'intermediate/train_merged_small.csv')
train_input_small = h2o.import_file(path_small_input)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [6]:
train_input_small

participant_id,EHQ_EHQ_Total,ColorVision_CV_Score,APQ_P_APQ_P_CP,APQ_P_APQ_P_ID,APQ_P_APQ_P_INV,APQ_P_APQ_P_OPD,APQ_P_APQ_P_PM,APQ_P_APQ_P_PP,SDQ_SDQ_Conduct_Problems,SDQ_SDQ_Difficulties_Total,SDQ_SDQ_Emotional_Problems,SDQ_SDQ_Externalizing,SDQ_SDQ_Generating_Impact,SDQ_SDQ_Hyperactivity,SDQ_SDQ_Internalizing,SDQ_SDQ_Peer_Problems,SDQ_SDQ_Prosocial,MRI_Track_Age_at_Scan,Basic_Demos_Enroll_Year,Basic_Demos_Study_Site,PreInt_Demos_Fam_Child_Ethnicity,PreInt_Demos_Fam_Child_Race,MRI_Track_Scan_Location,Barratt_Barratt_P1_Edu,Barratt_Barratt_P1_Occ,Barratt_Barratt_P2_Edu,Barratt_Barratt_P2_Occ,ADHD_Outcome,Sex_F
UmrK0vMLopoR,40,13,3,10,47,13,11,28,0,6,1,5,0,5,1,0,10,nan,2016,1,0,0,1,21,45,21,45,1,1
CPaeQkhcjg7d,-94.47,14,3,13,34,18,23,30,0,18,6,8,7,8,10,4,5,nan,2019,3,1,2,3,15,15,0,0,1,0
Nb4EetVPm3gs,-46.67,14,4,10,35,16,10,29,1,14,2,8,5,7,6,4,9,8.2399,2016,1,1,8,1,18,40,0,0,1,0
p4vPhVu91o4b,-26.68,10,5,12,39,19,16,28,6,24,4,16,9,10,8,4,6,nan,2018,3,0,8,3,15,30,18,0,1,1
M09PXs7arQ5E,0,14,5,15,40,20,24,28,1,18,4,11,4,10,7,3,9,8.94068,2019,3,0,1,3,15,20,0,0,1,1
tBGXkEdv2cp7,83.34,2,5,12,35,16,15,21,4,17,0,12,9,8,5,5,3,nan,2018,1,0,8,3,21,0,18,45,1,0
DgRP31gu21O9,94.47,14,3,14,27,18,20,21,0,9,1,5,1,5,4,3,10,16.7682,2018,3,0,0,3,18,25,15,35,1,0
ClMA0FwvFgLY,60,14,3,15,41,21,19,25,3,16,3,11,6,8,5,2,6,11.2213,2019,1,0,0,2,21,40,18,40,1,0
NVUkahaJ6fhf,73.34,14,4,16,46,23,15,25,5,23,6,14,2,9,9,3,10,8.57084,2015,1,0,0,1,21,40,18,35,1,1
u0JiZgdGuYvh,66.67,14,6,19,37,22,23,23,5,25,4,15,6,10,10,6,5,15.4102,2019,3,0,3,3,15,25,15,10,1,0


In [7]:
# Check response variable distribution
train_input_small["ADHD_Outcome"].table()

ADHD_Outcome,Count
0,382
1,831


In [8]:
train_input_small["Sex_F"].table()

Sex_F,Count
0,797
1,416


### Clean up data types

In [15]:
train_input_small.columns

['participant_id',
 'EHQ_EHQ_Total',
 'ColorVision_CV_Score',
 'APQ_P_APQ_P_CP',
 'APQ_P_APQ_P_ID',
 'APQ_P_APQ_P_INV',
 'APQ_P_APQ_P_OPD',
 'APQ_P_APQ_P_PM',
 'APQ_P_APQ_P_PP',
 'SDQ_SDQ_Conduct_Problems',
 'SDQ_SDQ_Difficulties_Total',
 'SDQ_SDQ_Emotional_Problems',
 'SDQ_SDQ_Externalizing',
 'SDQ_SDQ_Generating_Impact',
 'SDQ_SDQ_Hyperactivity',
 'SDQ_SDQ_Internalizing',
 'SDQ_SDQ_Peer_Problems',
 'SDQ_SDQ_Prosocial',
 'MRI_Track_Age_at_Scan',
 'Basic_Demos_Enroll_Year',
 'Basic_Demos_Study_Site',
 'PreInt_Demos_Fam_Child_Ethnicity',
 'PreInt_Demos_Fam_Child_Race',
 'MRI_Track_Scan_Location',
 'Barratt_Barratt_P1_Edu',
 'Barratt_Barratt_P1_Occ',
 'Barratt_Barratt_P2_Edu',
 'Barratt_Barratt_P2_Occ',
 'ADHD_Outcome',
 'Sex_F']

In [21]:
train_input_small['Sex_F'] = train_input_small['Sex_F'].asfactor()
train_input_small['ADHD_Outcome'] = train_input_small['ADHD_Outcome'].asfactor()

In [18]:
train_data_path = os.path.join(home_dir, 'data/TRAIN/')
cat_input = pd.read_excel(os.path.join(train_data_path, 'TRAIN_CATEGORICAL_METADATA.xlsx'))
cat_col = [c for c in cat_input.columns if c != 'participant_id']
cat_col

['Basic_Demos_Enroll_Year',
 'Basic_Demos_Study_Site',
 'PreInt_Demos_Fam_Child_Ethnicity',
 'PreInt_Demos_Fam_Child_Race',
 'MRI_Track_Scan_Location',
 'Barratt_Barratt_P1_Edu',
 'Barratt_Barratt_P1_Occ',
 'Barratt_Barratt_P2_Edu',
 'Barratt_Barratt_P2_Occ']

In [20]:
for c in cat_col:
    train_input_small[c] = train_input_small[c].asfactor()

In [22]:
train, test = train_input_small.split_frame([0.8], seed=42)

In [23]:
print("train:%d test:%d" % (train.nrows, test.nrows))

train:963 test:250


In [24]:
y1 = "ADHD_Outcome"
y2 = "Sex_F"
ignore = ["ADHD_Outcome", "Sex_F"] 
x = list(set(train.names) - set(ignore))

In [ ]:
aml = H2OAutoML(max_models=25, 
                max_runtime_secs_per_model=30, 
                seed=623, 
                project_name='classification', 
                balance_classes=False)
%time aml.train(x=x, y=y1, training_frame=train)

AutoML progress: |
16:53:40.247: _train param, Dropping bad and constant columns: [participant_id]

█
16:53:47.625: _train param, Dropping bad and constant columns: [participant_id]

███
16:53:53.124: _train param, Dropping bad and constant columns: [participant_id]
16:53:57.303: _train param, Dropping bad and constant columns: [participant_id]

█
16:53:59.342: _train param, Dropping bad and constant columns: [participant_id]

██
16:54:08.197: _train param, Dropping bad and constant columns: [participant_id]

█
16:54:10.899: _train param, Dropping bad and constant columns: [participant_id]
16:54:13.273: _train param, Dropping bad and constant columns: [participant_id]

██

In [ ]:
lb = aml.leaderboard
lb.head(rows=lb.nrows)